# A spare EMA of the student

In distillation the student is trained by the losses, and its weights jump around from
step to step: every SGD update follows one noisy batch. A **spare EMA** is a second
copy of the student whose weights are an exponential moving average of the trained
one, updated after every optimizer step:

```
ema <- m * ema + (1 - m) * student        (m close to 1, e.g. 0.98)
```

The EMA is never trained directly, so it averages out the noise of the last ~1 / (1 - m)
steps. It often generalises better than the raw student, and costs nothing to keep
apart from memory and one extra forward pass. It is the model we export at the end.

Nothing new is needed in `lwfm`: three existing pieces compose into it.

| Piece | Role |
|---|---|
| a frozen `ModelNode` named `student_ema` | holds the averaged weights; gets no gradients, stays in eval mode |
| `EMAUpdate("student", "student_ema")` | moves the EMA towards the student after every optimizer step |
| `ModelCheckpoint(node="student_ema")` + an output sink | saves the EMA's weights and records where they went |

1. The data
2. A teacher to distil from
3. The graph: teacher, student and its spare EMA
4. Losses and metrics
5. Training, with the EMA updated and exported
6. What was saved
7. Loading the exported EMA as a plain `nn.Module`
8. Notes

Run from `examples/`. `lwfm` isn't an installed package yet, so put the repository root on the import path.

In [1]:
import sys
from pathlib import Path

# Make `import lwfm` work from examples/ without installing it
sys.path.insert(0, str(Path.cwd().parent))

## 1. The data

The same synthetic task as the `Trainer` tutorial: four classes, each a fixed random
"prototype" image, and every sample a noisy copy of its class's prototype. The noise is
heavy, so single batches pull the student in different directions: exactly the noise
an EMA smooths out.

In [2]:
import torch

K = 4  # number of classes
# One fixed random 3x8x8 image per class
prototypes = torch.rand(K, 3, 8, 8, generator=torch.Generator().manual_seed(0))


def make_batches(count: int, batch_size: int = 32, seed: int = 0) -> list[dict]:
    """`count` batches of `batch_size` noisy prototype images with their class labels,
    in lwfm's batch layout. The same seed always gives the same batches."""
    generator = torch.Generator().manual_seed(seed)
    batches = []
    for _ in range(count):
        labels = torch.randint(0, K, (batch_size,), generator=generator)
        noise = torch.randn(batch_size, 3, 8, 8, generator=generator)
        # Prototype plus heavy noise (std 1.5), so the student stays noisy
        images = prototypes[labels] + 1.5 * noise
        # Read later as "inputs.image" and "targets.class"
        batches.append({"inputs": {"image": images}, "targets": {"class": labels}})
    return batches


# Any iterable of batches works as a loader; DataLoaders in real runs
train_loader = make_batches(20)  # 20 batches make one epoch
val_loader = make_batches(10, seed=1)  # new samples from the same classes

## 2. A teacher to distil from

A small MLP trained with plain cross-entropy. In a real run this would be a large
pretrained network; here it only needs to be clearly better than the student.

In [3]:
from torch import nn
from torchmetrics.classification import MulticlassAccuracy

from lwfm.core.graph import ExecutionGraph, ModelNode
from lwfm.core.losses import LossBinding, LossComposer
from lwfm.evaluation.metrics import MetricBinding
from lwfm.training import Trainer, seed_everything


def accuracy(name: str, node: str) -> MetricBinding:
    """Top-1 accuracy of `node`'s logits, reported as `name`."""
    return MetricBinding(
        name,
        MulticlassAccuracy(K, average="micro"),
        [f"{node}.logits", "targets.class"],
    )


seed_everything(0)  # before building the model: weight init draws from the seed
teacher_node = ModelNode(
    "teacher",
    nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, K)),
    inputs="inputs.image",
    outputs="logits",  # published as "teacher.logits"
)
teacher_result = Trainer(device="cpu", max_epochs=5).fit(
    ExecutionGraph([teacher_node]),
    train_loader,
    LossComposer(
        [LossBinding("ce", nn.CrossEntropyLoss(), ["teacher.logits", "targets.class"])]
    ),
    torch.optim.SGD(teacher_node.parameters(), lr=0.1),
    val_loader=val_loader,
    metrics=[accuracy("top1", "teacher")],
)
teacher_result.history[-1]  # (step, metrics) of the final validation

(100, {'top1': 0.890625})

## 3. The graph: teacher, student and its spare EMA

Three nodes read the same image:

```
inputs.image --+--> teacher      (frozen)  --> teacher.logits     --+
               |                                                    +--> loss: ce + kd
               +--> student      (trained) --> student.logits     --+
               |         |
               |         |  EMAUpdate, after every optimizer step
               |         v
               +--> student_ema  (frozen)  --> student_ema.logits ----> metric only
```

- The **student** is a cheaper model than the teacher: it sees the image at half
  resolution.
- The **EMA** starts as an exact copy of the student (`copy.deepcopy`), so the two
  have the same architecture, which `EMAUpdate` checks before training.
- `frozen=True` on the EMA means no gradients and eval mode throughout. It also means
  the EMA has no trainable parameters, so the optimizer is built over the student alone
  and the trainer's "every trainable parameter is in the optimizer" check still passes.

In [4]:
import copy


def make_student_module() -> nn.Module:
    """The student architecture: average-pool to 4x4, then a linear classifier."""
    return nn.Sequential(nn.AvgPool2d(2), nn.Flatten(), nn.Linear(3 * 4 * 4, K))


teacher = ModelNode(
    "teacher",
    copy.deepcopy(teacher_node.module),  # the trained weights, now frozen
    inputs="inputs.image",
    outputs="logits",
    frozen=True,  # no gradients, always in eval mode
)
seed_everything(0)
student = ModelNode(
    "student", make_student_module(), inputs="inputs.image", outputs="logits"
)
# Frozen, so only EMAUpdate ever changes its weights
student_ema = ModelNode(
    "student_ema",
    copy.deepcopy(student.module),  # same architecture and starting weights
    inputs="inputs.image",
    outputs="logits",
    frozen=True,
)
graph = ExecutionGraph([teacher, student, student_ema])
list(graph.nodes)

['teacher', 'student', 'student_ema']

## 4. Losses and metrics

The losses read only the teacher and the student: the EMA never contributes a
gradient. It is the standard Hinton loss,
`0.5 * CE(student, labels) + 0.5 * T^2 * KL(softmax(teacher/T) || softmax(student/T))`.

The metrics are where the EMA appears: validation scores the student, its EMA and the
teacher side by side, so we can see whether the EMA is actually better. `top1_ema` is
also what decides which EMA weights get saved.

In [5]:
import torch.nn.functional as F


class KDLoss(nn.Module):
    """Hinton distillation: T^2 * KL(softmax(teacher/T) || softmax(student/T))."""

    def __init__(self, temperature: float = 4.0):
        super().__init__()
        self.t = temperature

    def forward(self, student_logits, teacher_logits):
        s = F.log_softmax(student_logits / self.t, dim=-1)
        t = F.log_softmax(teacher_logits / self.t, dim=-1)
        # kl_div(s, t) is KL(t || s); T^2 keeps the gradient scale comparable to CE
        return F.kl_div(s, t, log_target=True, reduction="batchmean") * self.t**2


# Neither term reads student_ema, so the EMA never gets a gradient
loss = LossComposer(
    [
        LossBinding(
            "ce", nn.CrossEntropyLoss(), ["student.logits", "targets.class"], weight=0.5
        ),
        LossBinding("kd", KDLoss(), ["student.logits", "teacher.logits"], weight=0.5),
    ]
)
# Validation scores all three models side by side
metrics = [
    accuracy("top1", "student"),
    accuracy("top1_ema", "student_ema"),
    accuracy("top1_teacher", "teacher"),
]

## 5. Training, with the EMA updated and exported

Three callbacks do all the work. Order matters only within a hook, and here each
callback acts at a different point:

| Callback | Hook | Effect |
|---|---|---|
| `EMAUpdate("student", "student_ema", momentum=0.98)` | `on_after_optimizer_step` | `ema <- 0.98 * ema + 0.02 * student` |
| `ModelCheckpoint(node="student_ema", monitored_metric="top1_ema", subdir="checkpoints/ema", save_last=False)` | `on_validation_end` | saves the EMA's weights to `checkpoints/ema/best.pt` whenever `top1_ema` improves |
| `ModelCheckpoint(node="student")` | `on_fit_end` | saves the full training state, EMA included, to `checkpoints/last.pt`, and the student's final weights to `checkpoints/last_weights.pt` |

The two checkpoints use different `subdir`s; otherwise both would write
`checkpoints/best.pt` and one would overwrite the other.

**Choosing `m`.** The EMA averages over roughly `1 / (1 - m)` steps: 50 steps for
`m = 0.98`. The run is 160 optimizer steps, so the average covers about a third of it.
With `m = 0.999` (a common value for long ImageNet runs) the EMA would still be mostly
its random starting weights at the end of this short run.

The learning rate is deliberately high (0.5, with momentum), so the raw student is noisy
and the EMA's advantage is easy to see. `fit` runs inside `OutputRouter(...)`, which
passes every event, including the checkpoints' `artifact` events, to
`LocalFileSink`, and records the run's final status.

In [6]:
import tempfile

from lwfm.results.router import OutputRouter
from lwfm.results.sinks import LocalFileSink
from lwfm.training.callbacks import EMAUpdate, ModelCheckpoint

run_dir = Path(tempfile.mkdtemp()) / "kd_with_ema"  # a fresh directory per run

callbacks = [
    # student_ema <- 0.98 * student_ema + 0.02 * student, after every optimizer step
    EMAUpdate("student", "student_ema", momentum=0.98),
    # The exported model: the EMA's weights, whenever top1_ema improves
    ModelCheckpoint(
        run_dir,
        node="student_ema",
        monitored_metric="top1_ema",
        subdir="checkpoints/ema",  # apart from the student's files
        save_last=False,  # keep only the best EMA; last.pt below includes it too
    ),
    # When fit ends: the full state (last.pt) and the student's weights
    ModelCheckpoint(run_dir, node="student"),
]

# The router sends every event, checkpoints included, to the sink and records
# the run's final status
with OutputRouter(LocalFileSink(run_dir)) as sink:
    # 8 epochs x 20 batches = 160 optimizer steps
    result = Trainer(device="cpu", max_epochs=8, val_every_n_steps=20).fit(
        graph,
        train_loader,
        loss,
        # Only the student is optimised; the high lr makes it noisy on purpose
        torch.optim.SGD(student.parameters(), lr=0.5, momentum=0.9),
        val_loader=val_loader,
        metrics=metrics,
        callbacks=callbacks,
        output_sink=sink,
    )

print(f"{'step':>4}  {'student':>7}  {'ema':>7}  {'teacher':>7}")
for step, values in result.history:
    print(
        f"{step:>4}  {values['top1']:>7.3f}  {values['top1_ema']:>7.3f}"
        f"  {values['top1_teacher']:>7.3f}"
    )

step  student      ema  teacher
  20    0.575    0.600    0.891
  40    0.544    0.606    0.891
  60    0.522    0.609    0.891
  80    0.556    0.594    0.891
 100    0.550    0.594    0.891
 120    0.528    0.600    0.891
 140    0.519    0.609    0.891
 160    0.541    0.606    0.891


The raw student swings from one validation to the next; the EMA is steadier and higher,
because each of its weights averages the student's last ~50 updates. `result.best`
picks the validation where each model peaked:

In [7]:
# Each model's best validation; the EMA's is what ema/best.pt holds
result.best("top1"), result.best("top1_ema")

({'top1': 0.574999988079071,
  'top1_ema': 0.6000000238418579,
  'top1_teacher': 0.890625,
  'step': 20},
 {'top1': 0.5218750238418579,
  'top1_ema': 0.609375,
  'top1_teacher': 0.890625,
  'step': 60})

## 6. What was saved

The checkpoints write the files first, then send an `artifact` event holding the path
(relative to the run directory, so the directory can be moved). `LocalFileSink` records
those events in `events.jsonl` next to the scalars.

In [8]:
import json

print(sorted(str(path.relative_to(run_dir)) for path in run_dir.rglob("*.pt")))
# One artifact event per save; best.pt was overwritten at each improvement
for line in (run_dir / "events.jsonl").open():
    event = json.loads(line)
    if event["kind"] == "artifact":
        print(event["name"], "->", event["value"], "at step", event["step"])

['checkpoints/ema/best.pt', 'checkpoints/last.pt', 'checkpoints/last_weights.pt']
checkpoint/best -> checkpoints/ema/best.pt at step 20
checkpoint/best -> checkpoints/ema/best.pt at step 40
checkpoint/best -> checkpoints/ema/best.pt at step 60
checkpoint/last -> checkpoints/last.pt at step 160
checkpoint/last_weights -> checkpoints/last_weights.pt at step 160


`checkpoints/ema/best.pt` holds `deployable_model("student_ema").state_dict()`: the
student architecture's weights and nothing else, no teacher or graph. `last.pt` holds
the whole graph, so the final EMA weights are in there too, under the `student_ema`
node.

In [9]:
# weights_only=True loads tensors only, never arbitrary pickled code
ema_weights = torch.load(run_dir / "checkpoints/ema/best.pt", weights_only=True)
print("ema/best.pt:", list(ema_weights))  # "2." is the Linear layer

last = torch.load(run_dir / "checkpoints/last.pt", weights_only=True)
print(
    "last.pt EMA keys:", [k for k in last["graph"] if k.startswith("nodes.student_ema")]
)

ema/best.pt: ['2.weight', '2.bias']
last.pt EMA keys: ['nodes.student_ema.module.2.weight', 'nodes.student_ema.module.2.bias']


## 7. Loading the exported EMA as a plain `nn.Module`

The file holds weights, not code: build the student architecture, then load them into
it. From there it's an ordinary `nn.Module`, with no trace of the EMA machinery.
Evaluating it reproduces the best `top1_ema` from training.

In [10]:
from lwfm.evaluation.evaluator import Evaluator

exported = make_student_module()
exported.load_state_dict(ema_weights)  # strict: a mismatched key would raise
exported.eval()

# A one-node graph only to reuse the Evaluator; the node name is just a label
evaluation = Evaluator().evaluate(
    ExecutionGraph(
        [ModelNode("student", exported, inputs="inputs.image", outputs="logits")]
    ),
    val_loader,
    [accuracy("top1", "student")],
    device=torch.device("cpu"),
)
evaluation.metric_values  # equals the best top1_ema above

{'top1': 0.609375}

## 8. Notes

- **Extra compute.** `ExecutionGraph` runs every node on every batch, so the EMA does
  one forward pass (without gradients) per training batch even though no loss reads it.
  That is roughly a third of the cost of a student training step.
- **`m` depends on run length.** The EMA covers about `1 / (1 - m)` optimizer steps.
  Long runs use 0.999 to 0.9999; short runs need smaller values, or the EMA ends up
  dominated by the student's early weights.
- **BatchNorm.** `EMAUpdate` averages parameters but *copies* buffers (BatchNorm running
  statistics) from the student, since averaging running statistics isn't meaningful.
- **Fixed-schedule comparisons.** When every run in a comparison must keep its final
  weights rather than a hand-picked best epoch (e.g. kd_plan's E0 to E3), drop
  `monitored_metric` and keep `save_last=True` on the EMA checkpoint. It then writes
  `checkpoints/ema/last_weights.pt`: the final EMA weights alone, ready to load.
- **Same idea, other targets.** `EMAUpdate` is not specific to students: with a teacher
  as the target it gives mean-teacher / DINO-style self-distillation.